# 01 · 生成模型的世界觀：從加噪到去噪

歡迎來到 **生成式影像 → 擴散模型生成影像**。

前面的軌道讓模型**辨識**影像(分類、偵測)。這條軌道反過來:讓模型**創造**影像。能無中生有畫出一張圖的模型,叫**生成模型**。

## 三條路線的直覺

- **VAE**:學會把圖壓成一個小向量、再還原。生成 = 從向量空間取一點解碼。圖通常偏糊。
- **GAN**:一個「畫家」和一個「鑑定師」對抗訓練。圖很銳利,但訓練不穩、容易崩。
- **Diffusion(擴散)**:今天的主流(Stable Diffusion、Midjourney、DALL·E 都是)。點子優雅到不可思議——

> **先學會「把圖一步步加噪變成雪花」,再反過來「從雪花一步步去噪還原成圖」。** 會去噪,就會生成:餵一張純噪聲進去,反覆去噪,就「長」出一張全新的圖。

## 這堂課你會學到

- 三種生成模型(VAE / GAN / Diffusion)的核心直覺與取捨
- **親眼看擴散的「前向過程」**:把一張 MNIST 數字一步步加噪,直到變成純雪花
- 建立「**破壞是為了學會重建**」的心智模型——整條軌道的靈魂

## 1. 安裝與設定

In [ ]:
%pip install -q -U torchvision

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. 拿一張 MNIST 數字

In [ ]:
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

tf = transforms.Compose([transforms.Resize(32), transforms.ToTensor(),
                         transforms.Normalize((0.5,), (0.5,))])
mnist = torchvision.datasets.MNIST("./data", train=True, download=True, transform=tf)
x0, label = mnist[1]
print("一張數字,形狀", tuple(x0.shape), " 標籤", label)
plt.imshow(((x0[0] + 1) / 2), cmap="gray"); plt.axis("off"); plt.show()

## 3. 前向過程:一步步把它加噪成雪花

每一步混入一點高斯噪聲。看著數字慢慢溶解——擴散模型要學的,就是**反過來走這條路**。

In [ ]:
import torch

steps = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]      # 噪聲比例
fig, axes = plt.subplots(1, len(steps), figsize=(11, 2.2))
noise = torch.randn_like(x0)
for ax, s in zip(axes, steps):
    noisy = (1 - s) * x0 + s * noise          # 簡化示意:線性混入噪聲
    ax.imshow(((noisy[0].clamp(-1, 1) + 1) / 2), cmap="gray")
    ax.set_title(f"noise={s:.1f}", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

## 小結

- **生成模型讓 AI 創造而非辨識**。三大家族:VAE(糊但穩)、GAN(銳利但難訓)、Diffusion(現今主流)。
- 擴散的核心:**前向加噪**(圖 → 雪花)是固定的、不用學;要學的是**反向去噪**(雪花 → 圖)。
- 會去噪 → 餵純噪聲反覆去噪 → 生出全新的圖。

下一課:把這個「加噪」過程**寫成正式的數學排程**(forward diffusion)。